# TP3 — RAG multimodal agentique

**Sujet :** Pr. Mohamed YOUSSFI  
**Document :** Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (2017).

Ce notebook suit les sept étapes du TP : extraction texte/images, embeddings CLIP, index ChromaDB,
recherche cosinus, outil de recherche, agent ReAct avec LangGraph, puis validation.
La génération utilise un modèle **local** via Ollama ; aucune clé API n'est nécessaire.
Exécuter les cellules dans l'ordre. Le premier lancement télécharge le PDF et les poids CLIP ;
le modèle Ollama est téléchargé avec la commande de préparation ci-dessous.

## Préparation

Installer les dépendances de `requirements.txt`, démarrer Ollama et télécharger le modèle :

```powershell
python -m pip install -r requirements.txt
ollama pull qwen3-vl:4b
```

Dans VS Code, choisir le noyau Python où ces dépendances sont installées. Le modèle Qwen3-VL
traite les images et sait appeler des outils. Le dossier `artifacts/` est créé localement et ignoré par Git.
Si le téléchargement du PDF échoue, placer manuellement le PDF dans `pdfs/attention_is_all_you_need.pdf`.

In [ ]:
from pathlib import Path
from io import BytesIO
import base64
import hashlib
import json
import urllib.request

import chromadb
import open_clip
import pymupdf
import torch
import torch.nn.functional as F
from PIL import Image
from IPython.display import Markdown, display
from open_clip.tokenizer import SimpleTokenizer

# Fonctionne si VS Code démarre le noyau dans TP3 ou à la racine du dépôt.
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "TP3_RAG_multimodal_agentique.ipynb").is_file():
    candidate = BASE_DIR / "TP3"
    if (candidate / "TP3_RAG_multimodal_agentique.ipynb").is_file():
        BASE_DIR = candidate
    else:
        raise FileNotFoundError("Ouvrir le notebook depuis le dossier TP3 du dépôt.")

ARTIFACTS = BASE_DIR / "artifacts"
TEXT_DIR = ARTIFACTS / "textes"
IMAGE_DIR = ARTIFACTS / "images"
PDF_DIR = BASE_DIR / "pdfs"
for folder in (ARTIFACTS, TEXT_DIR, IMAGE_DIR, PDF_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dossier TP3 :", BASE_DIR)
print("Appareil CLIP :", DEVICE)

## Étape 1 — PDF, texte et images

Le PDF est identifié par son SHA-256. Pour chaque page, on sauvegarde son texte, les images intégrées
et un rendu PNG de la page. Les rendus conservent aussi les schémas vectoriels que `extract_image`
ne peut pas extraire comme images autonomes. Les numéros de page sont conservés pour les citations.

In [ ]:
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
candidates = [
    BASE_DIR / "Attention is all you need.pdf",
    PDF_DIR / "attention is all you need.pdf",
    PDF_DIR / "attention_is_all_you_need.pdf",
]
PDF_PATH = next((path for path in candidates if path.is_file()), candidates[-1])

if not PDF_PATH.is_file():
    request = urllib.request.Request(PAPER_URL, headers={"User-Agent": "TP3-RAG/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=45) as response:
            pdf_bytes = response.read()
        if not pdf_bytes.startswith(b"%PDF-"):
            raise ValueError("Le téléchargement n'a pas renvoyé un PDF.")
        PDF_PATH.write_bytes(pdf_bytes)
    except Exception as error:
        raise RuntimeError(
            "Télécharger https://arxiv.org/pdf/1706.03762 puis enregistrer le fichier "
            f"dans {PDF_PATH} avant de relancer cette cellule."
        ) from error

DOCUMENT_ID = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
print("PDF :", PDF_PATH)
print("SHA-256 :", DOCUMENT_ID)

In [ ]:
pages_texte = []
images_pdf = []
with pymupdf.open(PDF_PATH) as document:
    for numero, page in enumerate(document, start=1):
        texte = page.get_text("text", sort=True)
        texte_path = TEXT_DIR / f"page_{numero:03d}.txt"
        texte_path.write_text(texte, encoding="utf-8")
        pages_texte.append({"page": numero, "texte": texte, "chemin": texte_path})

        for xref in sorted({info[0] for info in page.get_images(full=True)}):
            data = document.extract_image(xref)
            if not data:
                continue
            try:
                with Image.open(BytesIO(data["image"])) as source:
                    image = source.convert("RGB")
                    chemin = IMAGE_DIR / f"page_{numero:03d}_image_{xref}.png"
                    image.save(chemin)
                images_pdf.append({"page": numero, "chemin": chemin, "nature": "image_extraite"})
            except (OSError, ValueError) as error:
                print(f"Image xref {xref} de la page {numero} ignorée : {error}")

        rendu_path = IMAGE_DIR / f"page_{numero:03d}_rendu.png"
        page.get_pixmap(matrix=pymupdf.Matrix(1.5, 1.5), alpha=False).save(rendu_path)
        images_pdf.append({"page": numero, "chemin": rendu_path, "nature": "rendu_page"})

print(f"{len(pages_texte)} pages, {len(images_pdf)} images ou rendus enregistrés.")
print("Première page (extrait) :", pages_texte[0]["texte"][:300].replace("\n", " "))

In [ ]:
print("Exemple de rendu :", images_pdf[0]["chemin"].name)
with Image.open(images_pdf[0]["chemin"]) as image:
    apercu = image.copy()
    apercu.thumbnail((500, 650))
    display(apercu)

## Étape 2 — Espace latent commun CLIP

`ViT-B-32` encode texte et images dans le même espace. Le texte est découpé en fragments de
60 tokens BPE avec chevauchement de 10, sous la limite de contexte de CLIP. Chaque vecteur est
normalisé à une norme L2 égale à 1 avant indexation et avant recherche.

In [ ]:
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED
)
clip_model = clip_model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
bpe = SimpleTokenizer()
print("CLIP chargé :", MODEL_NAME, PRETRAINED)

In [ ]:
def fragments_clip(texte: str, taille: int = 60, chevauchement: int = 10):
    if not 0 <= chevauchement < taille <= 75:
        raise ValueError("Choisir 0 <= chevauchement < taille <= 75.")
    tokens = bpe.encode(texte)
    for debut in range(0, len(tokens), taille - chevauchement):
        fragment = bpe.decode(tokens[debut:debut + taille]).strip()
        if fragment:
            yield fragment
        if debut + taille >= len(tokens):
            break


@torch.inference_mode()
def encoder_textes(textes: list[str]) -> list[list[float]]:
    if not textes:
        return []
    tokens = tokenizer(textes).to(DEVICE)
    vecteurs = clip_model.encode_text(tokens).float()
    return F.normalize(vecteurs, p=2, dim=-1).cpu().tolist()


@torch.inference_mode()
def encoder_image(chemin: str | Path) -> list[float]:
    chemin = Path(chemin)
    if not chemin.is_absolute():
        chemin = BASE_DIR / chemin
    with Image.open(chemin) as image:
        pixels = preprocess(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    vecteur = clip_model.encode_image(pixels).float()
    return F.normalize(vecteur, p=2, dim=-1)[0].cpu().tolist()


exemple_texte = encoder_textes(["a diagram of the Transformer architecture"])[0]
exemple_image = encoder_image(images_pdf[0]["chemin"])
assert len(exemple_texte) == len(exemple_image)
assert abs(sum(x * x for x in exemple_texte) - 1.0) < 1e-4
assert abs(sum(x * x for x in exemple_image) - 1.0) < 1e-4
print("Dimension commune et normes vérifiées :", len(exemple_texte))

## Étape 3 — Indexation ChromaDB

Une seule collection stocke les vecteurs de texte et d'images. La distance est cosinus.
Avant de remplacer l'index du même PDF, tous ses anciens éléments sont supprimés. Les métadonnées
incluent le type, le numéro de page, le chemin relatif et l'identifiant SHA-256 du document.

In [ ]:
client = chromadb.PersistentClient(path=str(ARTIFACTS / "chroma"))
collection = client.get_or_create_collection(
    name="tp3_clip_vit_b32_laion2b",
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
)
print("Collection :", collection.name, "| éléments avant indexation :", collection.count())

In [ ]:
def chemin_relatif(chemin: Path) -> str:
    return chemin.relative_to(BASE_DIR).as_posix()


elements = []
for page in pages_texte:
    for numero_fragment, fragment in enumerate(fragments_clip(page["texte"])):
        elements.append({
            "id": f"{DOCUMENT_ID}:texte:{page['page']}:{numero_fragment}",
            "document": fragment,
            "metadata": {
                "type": "texte", "page": page["page"],
                "source": chemin_relatif(page["chemin"]),
                "source_name": PDF_PATH.name, "document_id": DOCUMENT_ID,
            },
        })

for image in images_pdf:
    elements.append({
        "id": f"{DOCUMENT_ID}:image:{image['chemin'].name}",
        "document": f"{image['nature']} du PDF, page {image['page']}",
        "metadata": {
            "type": "image", "page": image["page"],
            "source": chemin_relatif(image["chemin"]),
            "nature": image["nature"], "source_name": PDF_PATH.name,
            "document_id": DOCUMENT_ID,
        },
    })

# Calculer les vecteurs avant de supprimer l'index précédent.
for debut in range(0, len(elements), 32):
    lot = elements[debut:debut + 32]
    textes = [item for item in lot if item["metadata"]["type"] == "texte"]
    for item, vecteur in zip(textes, encoder_textes([x["document"] for x in textes])):
        item["embedding"] = vecteur
    for item in lot:
        if item["metadata"]["type"] == "image":
            item["embedding"] = encoder_image(item["metadata"]["source"])

assert elements and len({item["id"] for item in elements}) == len(elements)
collection.delete(where={"source_name": PDF_PATH.name})
for debut in range(0, len(elements), 64):
    lot = elements[debut:debut + 64]
    collection.upsert(
        ids=[item["id"] for item in lot],
        documents=[item["document"] for item in lot],
        metadatas=[item["metadata"] for item in lot],
        embeddings=[item["embedding"] for item in lot],
    )

assert collection.count() == len(elements)
print(f"Index mis à jour : {len(elements)} éléments, dont {len(images_pdf)} images/rendus.")